# Stage 5 Protocol Audit

Проверка train/validation split и подготовка protocol note. Сначала скопируйте `drop_in/scripts/audit_train_val_protocol.py` в `scripts/` репозитория.

In [ ]:

from pathlib import Path
import subprocess, json, os, shutil, csv
import pandas as pd
from IPython.display import display, Markdown

REPO_URL = os.environ.get("REPO_URL", "https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git")
REPO_DIR = Path(os.environ.get("REPO_DIR", "/kaggle/working/vlm-for-insulator-defect-detection"))
DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/kostyaryazanov/idid-coco-v3"),
    Path("/kaggle/input/idid-coco-v3"),
    Path("/kaggle/input"),
]
TRAIN_REL = Path("stage3_regrouped_v2/train_balanced/vlm_labels_v1_train_balanced_v2.annotated.jsonl")
VAL_REL = Path("stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl")

def sh(args, cwd=None, check=True):
    print("$", " ".join(str(a) for a in args))
    p = subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed {p.returncode}: {' '.join(str(a) for a in args)}")
    return p

def find_data_root():
    for root in DATASET_ROOT_CANDIDATES:
        if (root/TRAIN_REL).exists() and (root/VAL_REL).exists():
            return root
    for root in Path('/kaggle/input').glob('**') if Path('/kaggle/input').exists() else []:
        if (root/TRAIN_REL).exists() and (root/VAL_REL).exists():
            return root
    raise FileNotFoundError('Could not find stage3_regrouped_v2 train/val JSONL')


In [ ]:
DATA_ROOT = find_data_root()
TRAIN_JSONL = DATA_ROOT / TRAIN_REL
VAL_JSONL = DATA_ROOT / VAL_REL
print('DATA_ROOT:', DATA_ROOT)
print('TRAIN_JSONL:', TRAIN_JSONL)
print('VAL_JSONL:', VAL_JSONL)


In [ ]:
if not REPO_DIR.exists():
    sh(['git','clone',REPO_URL,str(REPO_DIR)])
print('Repo:', REPO_DIR)
sh(['python','-m','pip','install','-q','pyyaml','pandas'], cwd=REPO_DIR)


## Запуск audit

In [ ]:
sh([
    'python','scripts/audit_train_val_protocol.py',
    '--train-jsonl', TRAIN_JSONL,
    '--val-jsonl', VAL_JSONL,
    '--out-dir', 'reports/next_research/protocol',
], cwd=REPO_DIR)
print((REPO_DIR/'reports/next_research/protocol/train_val_protocol.md').read_text(encoding='utf-8'))


In [ ]:
dist = pd.read_csv(REPO_DIR/'reports/next_research/protocol/train_val_distribution.csv')
display(dist)
